In [1]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 56.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import pathlib
from transformers import VivitImageProcessor, VivitForVideoClassification
from PIL import Image
from torch.utils.data import Dataset
import numpy as np
import random
import av
from importlib.resources import path
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import time

2026-03-26 17:07:25.836302: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774544846.044713      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774544846.105813      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774544846.592211      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774544846.592244      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774544846.592247      24 computation_placer.cc:177] computation placer alr

In [3]:
dataset_root_path = pathlib.Path("/kaggle/input/wlasl300/wlasl300")

video_count_train = len(list(dataset_root_path.glob("train/*/*.mp4")))
video_count_val = len(list(dataset_root_path.glob("val/*/*.mp4")))
video_count_test = len(list(dataset_root_path.glob("test/*/*.mp4")))
video_total = video_count_train + video_count_val + video_count_test
print(f"Total videos: {video_total}")

all_video_file_paths = (
    list(dataset_root_path.glob("train/*/*.mp4"))
    + list(dataset_root_path.glob("val/*/*.mp4"))
    + list(dataset_root_path.glob("test/*/*.mp4"))
)
all_video_file_paths[:5]

class_labels = sorted({path.parent.name for path in all_video_file_paths})

label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"Unique classes ({len(class_labels)}): {class_labels}")

Total videos: 5118
Unique classes (300): ['about', 'accident', 'africa', 'again', 'all', 'always', 'animal', 'apple', 'approve', 'argue', 'arrive', 'baby', 'back', 'backpack', 'bad', 'bake', 'balance', 'ball', 'banana', 'bar', 'basketball', 'bath', 'bathroom', 'beard', 'because', 'bed', 'before', 'behind', 'bird', 'birthday', 'black', 'blanket', 'blue', 'book', 'bowling', 'boy', 'bring', 'brother', 'brown', 'business', 'but', 'buy', 'call', 'can', 'candy', 'careful', 'cat', 'catch', 'center', 'cereal', 'chair', 'champion', 'change', 'chat', 'cheat', 'check', 'cheese', 'children', 'christmas', 'city', 'class', 'clock', 'close', 'clothes', 'coffee', 'cold', 'college', 'color', 'computer', 'convince', 'cook', 'cool', 'copy', 'corn', 'cough', 'country', 'cousin', 'cow', 'crash', 'crazy', 'cry', 'cute', 'dance', 'dark', 'daughter', 'day', 'deaf', 'decide', 'delay', 'delicious', 'different', 'disappear', 'discuss', 'divorce', 'doctor', 'dog', 'door', 'draw', 'dress', 'drink', 'drive', 'drop'

In [4]:
image_processor = VivitImageProcessor.from_pretrained("Shawon16/ViViT_wlasl_100_200ep_coR_")
model = VivitForVideoClassification.from_pretrained(
    "Shawon16/ViViT_wlasl_100_200ep_coR_",
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

DEVICE = torch.device("cuda")
model.to(DEVICE)
print(next(model.parameters()).device)

preprocessor_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

Some weights of VivitForVideoClassification were not initialized from the model checkpoint at Shawon16/ViViT_wlasl_100_200ep_coR_ and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([100]) in the checkpoint and torch.Size([300]) in the model instantiated
- classifier.weight: found shape torch.Size([100, 768]) in the checkpoint and torch.Size([300, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cuda:0


In [5]:
class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, image_processor, num_frames, train=True):
        self.video_paths = video_paths
        self.labels = labels
        self.image_processor = image_processor
        self.num_frames = num_frames
        self.train = train

    def __len__(self):
        return len(self.video_paths)

    def _load_video(self, path):
        container = av.open(str(path))
        frames = []

        for i, frame in enumerate(container.decode(video=0)):
            img = Image.fromarray(frame.to_rgb().to_ndarray())
            img = img.resize((224, 224))
            frames.append(np.array(img))


        container.close()
        return frames

    def _sample_frames(self, frames):
        idx = np.linspace(0, len(frames) - 1, self.num_frames).astype(int)
        return [frames[i] for i in idx]

    def __getitem__(self, idx):
        frames = self._load_video(self.video_paths[idx])
        frames = self._sample_frames(frames)

        inputs = self.image_processor(
            frames,
            return_tensors="pt"
        )

        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = torch.tensor(self.labels[idx])

        return inputs

In [6]:
train_paths = list((dataset_root_path / "train").rglob("*.mp4"))
val_paths = list((dataset_root_path / "val").rglob("*.mp4"))
test_paths = list((dataset_root_path / "test").rglob("*.mp4"))

train_labels = [label2id[p.parent.name] for p in train_paths]
val_labels = [label2id[p.parent.name] for p in val_paths]
test_labels = [label2id[p.parent.name] for p in test_paths]

num_frames = model.config.num_frames

train_dataset = VideoDataset(
    train_paths,
    train_labels,
    image_processor,
    num_frames,
    train=True,
)

val_dataset = VideoDataset(
    val_paths,
    val_labels,
    image_processor,
    num_frames,
    train=False,
)

test_dataset = VideoDataset(
    test_paths,
    test_labels,
    image_processor,
    num_frames,
    train=False,
)


train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2)
test_loader = DataLoader(test_dataset, batch_size=2)

In [7]:
EPOCHS = 30
LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM_STEPS = 8
PATIENCE = 5
MIN_IMPR = 0.001

SAVE_PATH = "/kaggle/working/vivit300-finetuned.pth"

In [8]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scaler = torch.amp.GradScaler('cuda') 

best_val_loss = float('inf')
patience_counter = 0

start = time.time()
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 30)

    model.train()
    train_loss = 0
    optimizer.zero_grad()

    train_preds = []
    train_labels = []

    for step, batch in enumerate(tqdm(train_loader)):
        inputs = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            logits = outputs.logits
            loss = criterion(logits, labels)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * GRAD_ACCUM_STEPS

        preds = torch.argmax(logits, dim=1)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(labels.detach().cpu().numpy())

    train_acc = accuracy_score(train_labels, train_preds)
    avg_train_loss = train_loss / len(train_loader)

    
    model.eval()
    val_preds = []
    val_labels = []
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader):
            inputs = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                logits = outputs.logits
                loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Val Loss  : {avg_val_loss:.4f}")
    print(f"Val Acc   : {val_acc:.4f}")

    
    if avg_val_loss < best_val_loss - MIN_IMPR:
        print("Validation loss improved.")
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
    else:
        patience_counter += 1
        print(f"No significant improvement. Patience {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print("Training complete.")
print(f"Training time: {time.time() - start:.4f}")


model.load_state_dict(torch.load(SAVE_PATH))


Epoch 1/30
------------------------------


100%|██████████| 451/451 [04:14<00:00,  1.77it/s]


Train Loss: 5.6229
Train Acc : 0.0237
Val Loss  : 5.3310
Val Acc   : 0.0666
Validation loss improved.

Epoch 2/30
------------------------------


100%|██████████| 451/451 [04:12<00:00,  1.79it/s]


Train Loss: 4.7885
Train Acc : 0.2046
Val Loss  : 4.8052
Val Acc   : 0.1698
Validation loss improved.

Epoch 3/30
------------------------------


...

100%|██████████| 451/451 [04:10<00:00,  1.80it/s]


Train Loss: 0.9391
Train Acc : 0.9952
Val Loss  : 3.5630
Val Acc   : 0.4118
No significant improvement. Patience 4/5

Epoch 16/30
------------------------------


100%|██████████| 451/451 [04:12<00:00,  1.79it/s]

Train Loss: 0.9340
Train Acc : 0.9958
Val Loss  : 3.5679
Val Acc   : 0.4162
No significant improvement. Patience 5/5
Early stopping triggered.
Training complete.
Training time: 32308.7044


<All keys matched successfully>

In [9]:
model.eval()
test_preds = []
test_labels_list = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        inputs = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            logits = outputs.logits

        preds = torch.argmax(logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels_list.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_labels_list, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

100%|██████████| 334/334 [03:09<00:00,  1.76it/s]

Test Accuracy: 0.3698
